In [2]:
import pandas as pd

Data menunjuka informasi yang cukup bagus untuk kebutuhan insight terkait battery capacity, charging time

In [3]:
df = pd.read_csv("/Users/mvvkur/Documents/backend-ev-flow/evflow-fullstack/backend-ev-flow/data/raw/kaggle_indonesia_ev/indonesia_ev_specs_pricing_2026.csv")

df.head(10)

,Vehicle Name,Vehicle Price Range,Battery Capacity,Range (Jarak Tempuh),Power/Horsepower,Seating Capacity,Charging time,Source URL,Is EV
0,Wuling Air EV,"Rp 214 - 307,5 Juta",26.7 kWh,200 - 300 km,40 hp,4 Kursi,8.5 Jam,https://www.oto.com/mobil-baru/wuling/ev,True
1,VinFast VF 5,"Rp 323,17 Juta",29.6 kWh,260 km,94 hp,5 Kursi,30 Menit,https://www.oto.com/mobil-baru/vinfast/vf-5,True
2,MG 4 EV,Rp 345 - 419 Juta,51 - 64 kWh,425 km,168 - 201 hp,5 Kursi,35 Menit,https://www.oto.com/mobil-baru/mg/4-ev,True
3,BYD M6,Rp 383 - 433 Juta,55.4 - 71.8 kWh,420 - 530 km,154 hp,6 - 7 Kursi,40 Menit,https://www.oto.com/mobil-baru/byd/m6,True
4,Wuling Binguo EV,Rp 279 - 332 Juta,31.9 - 37.9 kWh,333 - 410 km,67 hp,5 Kursi,35 Menit,https://www.oto.com/mobil-baru/wuling/binguo-ev,True
5,VinFast VF e34,"Rp 411,86 Juta",41.9 kWh,318 km,147 hp,5 Kursi,27 Menit,https://www.oto.com/mobil-baru/vinfast/vf-e34,True
6,Wuling Cloud EV,Rp 415 - 443 Juta,50.6 kWh,460 km,134 hp,5 Kursi,30 Menit,https://www.oto.com/mobil-baru/wuling/cloud,True
7,BYD Seal,Rp 639 - 750 Juta,82.56 kWh,580 - 650 km,313 - 530 hp,5 Kursi,37 Menit,https://www.oto.com/mobil-baru/byd/seal,True
8,BYD Dolphin,Rp 369 - 429 Juta,44.9 - 60.48 kWh,410 - 490 km,95 - 204 hp,5 Kursi,30 Menit,https://www.oto.com/mobil-baru/byd/dolphin,True
9,CHERY E5,"Rp 379,9 - 419,9 Juta",61 kWh,430 km,208 hp,5 Kursi,28 Menit,https://www.oto.com/mobil-baru/chery/e5,True


## PLN SPKLU (petaspklu)

Dataset resmi PLN — 3.029 stasiun SPKLU se-Indonesia (snapshot Mei 2026). Field utama: `nama_lokasi`, `alamat`, `provinsi`, `kabupaten_kota`, `latitude`/`longitude`, `watt`, `type_charge`, `total_konektor`, plus detail per-chargerbox di kolom `chargerboxes`. Pembersihan mengikuti pipeline produksi (`api/sources.py`): koordinat wajib numerik dan bukan (0,0).

In [4]:
import json

PLN_PATH = "/Users/mvvkur/Documents/backend-ev-flow/evflow-fullstack/backend-ev-flow/data/raw/_petaspklu_all.json"

with open(PLN_PATH) as f:
    pln_raw = json.load(f)

pln = pd.DataFrame(pln_raw)

# lat/lon tersimpan sebagai string -> numeric, lalu buang koordinat tidak valid / (0,0)
pln["latitude"] = pd.to_numeric(pln["latitude"], errors="coerce")
pln["longitude"] = pd.to_numeric(pln["longitude"], errors="coerce")
pln = pln[pln["latitude"].notna() & pln["longitude"].notna()]
pln = pln[~((pln["latitude"] == 0) & (pln["longitude"] == 0))]

# "22 kW" -> 22.0 (ikut cara parsing di api/sources.py)
pln["power_kw"] = pd.to_numeric(pln["watt"].astype(str).str.split().str[0], errors="coerce")

print(f"{len(pln_raw):,} baris mentah -> {len(pln):,} stasiun dengan koordinat valid")
pln.head(10)

3,029 baris mentah -> 3,029 stasiun dengan koordinat valid


,id,provinsi,kabupaten_kota,nama_lokasi,alamat,latitude,longitude,keterangan,status,type_charge,watt,total_charger,total_konektor,chargerboxes,power_kw
0,1,DKI Jakarta,Kota ADM Jakarta Pusat,SPKLU PLN UID JAKARTA RAYA,"Jl. M.I. Ridwan Rais No.1, Gambir",-6.180390,106.833191,None,1,medium,22 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'C...",22
1,2,Banten,Kab Tangerang,SPKLU AEON Mall BSD,"Jl. BSD Raya Utama, Pagedangan",-6.304080,106.643679,None,1,medium,22 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'H...",22
2,3,Banten,Kab Tangerang,SPKLU Supermall Karawaci,"Jl. Boulevard Diponegoro, Bencongan",-6.225712,106.607328,None,1,medium,25 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'A...",25
3,4,Banten,Kota Tangerang,SPKLU Tangcity,"Jl. Jenderal Sudirman No.1, Cikokol",-6.193182,106.633549,None,1,medium,25 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'C...",25
4,5,Sumatera Selatan,Kota Palembang,SPKLU PLN UIW S2JB Rivai,Jalan Kapt. Arivai no. 37 Palembang,-2.979122,104.748281,None,1,ultrafast,180 kW,0,0,"[{'type_charge': 'ultrafast', 'chargerbox_id':...",180
5,6,Sumatera Utara,Kota Medan,SPKLU PLN ULP Medan Kota,Jl. Listrik No. 8 Medan,3.585722,98.676141,None,1,medium,30 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'A...",30
6,7,Lampung,Kab Lampung Selatan,SPKLU Rest Area KM 20 B TRANS-SUMATERA,"Rawi, Kec. Penengahan, Lampung",-5.725474,105.668960,None,1,ultrafast,100 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'D...",100
7,8,Jawa Barat,Kab Cirebon,SPKLU REST AREA (TRAVOY) KM 207 A Ruas Palikanci,"Rest Area KM 207, Setupatok, Kec. Mundu, Cireb...",-6.775515,108.562221,None,1,fast,50 kW,0,0,"[{'type_charge': 'fast', 'chargerbox_id': 'ABB...",50
8,9,Bali,Kota Denpasar,SPKLU PLN ULP DENPASAR,JL. P.B. Sudirman,-8.662908,115.217891,None,1,medium,30 kW,0,0,"[{'type_charge': 'medium', 'chargerbox_id': 'A...",30
9,10,Jawa Tengah,Kab Sragen,SPKLU REST AREA (TRAVOY) KM 519 A Ruas Solo-Ngawi,"Dusun III, Karang Malang, Masaran",-7.480695,110.919125,None,1,fast,50 kW,0,0,"[{'type_charge': 'fast', 'chargerbox_id': 'CSE...",50


In [5]:
# Sebaran stasiun per provinsi dan tipe charger
display(pln["provinsi"].str.strip().value_counts().head(10))
pln["type_charge"].value_counts(dropna=False)

provinsi
DKI Jakarta         731
Jawa Barat          539
Banten              295
Jawa Timur          245
Jawa Tengah         220
Bali                116
Sumatera Utara      104
Sumatera Barat       77
Kalimantan Timur     68
Sulawesi Selatan     57
Name: count, dtype: int64

type_charge
medium       1848
fast          451
standard      382
ultrafast     348
Name: count, dtype: int64

## Open Charge Map (OCM)

527 POI Jabodetabek, diambil via REST API `api.openchargemap.io/v3/poi` dengan parameter `boundingbox` (server-side filter), `maxresults=5000`, autentikasi `X-API-Key`. Lisensi **CC-BY-4.0** — atribusi ke Open Charge Map contributors. Nilai tambah vs PLN: data komunitas dengan detail per-connection (`PowerKW`, `Quantity`), tanggal verifikasi (`DateLastVerified`), dan status operasional (`StatusTypeID`, 50 = Operational).

In [6]:
OCM_PATH = "/Users/mvvkur/Documents/backend-ev-flow/evflow-fullstack/backend-ev-flow/data/raw/ocm_jakarta.json"

with open(OCM_PATH) as f:
    ocm_raw = json.load(f)

ocm = pd.json_normalize(ocm_raw)[[
    "ID", "AddressInfo.Title", "AddressInfo.Latitude", "AddressInfo.Longitude",
    "AddressInfo.Town", "AddressInfo.StateOrProvince",
    "NumberOfPoints", "StatusTypeID", "DateLastVerified",
]].rename(columns={
    "AddressInfo.Title": "name",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "AddressInfo.Town": "city",
    "AddressInfo.StateOrProvince": "province",
    "NumberOfPoints": "n_points",
    "StatusTypeID": "status_type_id",
    "DateLastVerified": "date_verified",
})

# power_kw = daya maksimum antar-connection (ikut api/sources.py);
# OCM satu-satunya sumber dengan detail per-connection (PowerKW, Quantity)
ocm["power_kw"] = [
    max([c.get("PowerKW") for c in (p.get("Connections") or []) if c.get("PowerKW")] or [float("nan")])
    for p in ocm_raw
]
ocm = ocm[ocm["latitude"].notna() & ocm["longitude"].notna()]

print(f"{len(ocm_raw):,} POI -> {len(ocm):,} dengan koordinat valid")
ocm.head(10)

527 POI -> 527 dengan koordinat valid


,ID,name,latitude,longitude,city,province,n_points,status_type_id,date_verified,power_kw
0,479002,PLN ULP Babelan,-6.184700,107.038800,Bekasi,Jawa Barat,1,50,2026-04-04T18:04:00Z,25
1,480817,Marketing Summarecon Crown Gading,-6.158745,106.999004,Bekasi,Jawa Barat,1,50,2026-04-04T18:01:00Z,180
2,479003,KBN Cakung,-6.144100,106.936100,Jakarta Utara,DKI Jakarta,1,50,2026-04-04T18:10:00Z,50
3,480821,Grand Waterfront Agung Sedayu,-6.166220,106.923050,Jakarta Timur,DKI Jakarta,1,50,2026-04-04T18:26:00Z,120
4,480819,Gading Festival,-6.169028,106.925170,Jakarta Timur,DKI Jakarta,1,50,2026-04-04T18:09:00Z,50
5,475456,(HVT) Old Shanghai Sedayu City,-6.165541,106.924533,Jakarta Timur,DKI Jakarta,1,50,2026-05-25T10:06:00Z,200
6,475463,BYD Arista Kelapa Gading,-6.164844,106.929264,NaN,NaN,1,50,2026-03-12T22:42:00Z,180
7,270735,Aeon Mall Jakarta Garden City,-6.172188,106.952221,Jakarta Timur,Jakarta,2,50,2026-03-12T22:42:00Z,60
8,270727,Hyundai Pegangsaan,-6.174608,106.913916,Jakarta Utara,Jakarta,1,50,2023-07-04T13:49:00Z,7
9,480820,Apartemen East Park,-6.202522,106.925422,Jakarta Timur,DKI Jakarta,1,50,2026-04-04T18:13:00Z,100


In [7]:
# Sebaran kota, status operasional, dan distribusi daya
display(ocm["city"].value_counts().head(10))
display(ocm["status_type_id"].value_counts(dropna=False))  # 50 = Operational
ocm["power_kw"].describe()

city
Jakarta Selatan      100
Tangerang Selatan     48
Jakarta Timur         45
Jakarta Pusat         45
Tangerang             43
Jakarta Utara         41
Bekasi                37
Jakarta Barat         32
Depok                 26
Bogor                 21
Name: count, dtype: int64

status_type_id
50     521
100      5
150      1
Name: count, dtype: int64

count    527.000000
mean      76.330171
std       64.887158
min        7.000000
25%       22.000000
50%       60.000000
75%      120.000000
max      480.000000
Name: power_kw, dtype: float64

## OpenStreetMap (Overpass API)

13 node `amenity=charging_station` di bbox Jabodetabek, diambil via query Overpass (`overpass-api.de/api/interpreter`), snapshot 2026-05-28. Lisensi **ODbL** — © OpenStreetMap contributors (tercantum eksplisit di payload `osm3s.copyright`). Tag OSM jarang lengkap: nama pakai rantai fallback `name → brand → operator → ref`, daya dari `charging_station:output`. Sumber ini kecil tapi independen — menangkap SPKLU rest area tol yang tidak ada di sumber lain.

In [8]:
OSM_PATH = "/Users/mvvkur/Documents/backend-ev-flow/evflow-fullstack/backend-ev-flow/data/raw/osm_charging_jakarta.json"

with open(OSM_PATH) as f:
    osm_payload = json.load(f)

def _kw(v):
    try:
        return float(str(v).split()[0])
    except (TypeError, ValueError):
        return float("nan")

rows = []
for el in osm_payload.get("elements", []):
    tags = el.get("tags", {})
    lat = el.get("lat") or (el.get("center") or {}).get("lat")
    lon = el.get("lon") or (el.get("center") or {}).get("lon")
    if lat is None or lon is None:
        continue
    rows.append({
        "id": f"osm-{el['type']}-{el['id']}",
        # banyak node SPKLU hanya punya brand/operator/ref, bukan name (ikut api/sources.py)
        "name": tags.get("name") or tags.get("name:en") or tags.get("brand")
                or tags.get("operator") or tags.get("network") or tags.get("ref"),
        "latitude": lat,
        "longitude": lon,
        "operator": tags.get("operator") or tags.get("brand") or tags.get("network"),
        "power_kw": _kw(tags.get("charging_station:output") or tags.get("socket:type2_combo:output")),
        "capacity": tags.get("capacity"),
        "tags": tags,
    })

osm = pd.DataFrame(rows)
print(f"Snapshot OSM {osm_payload['osm3s']['timestamp_osm_base']} -> {len(osm)} stasiun")
osm.drop(columns=["tags"]).head(13)

Snapshot OSM 2026-05-28T09:34:35Z -> 13 stasiun


,id,name,latitude,longitude,operator,power_kw,capacity
0,osm-node-6222913188,BPPT,-6.184263,106.821790,BPPT,NaN,NaN
1,osm-node-12077015474,SPKLU Rest Area KM 19B,-6.271266,107.039482,NaN,NaN,NaN
2,osm-node-12077015475,SPKLU Rest Area KM 6,-6.259146,106.928444,NaN,NaN,NaN
3,osm-node-12557877196,NaN,-6.248216,106.990448,NaN,NaN,NaN
4,osm-node-12621583669,SPKLU PLN,-6.181879,106.824665,SPKLU PLN,NaN,NaN
5,osm-node-12650943353,NaN,-6.180317,106.832814,NaN,NaN,NaN
6,osm-node-12960379004,Carzo,-6.220864,106.653811,NaN,NaN,NaN
7,osm-node-13081235461,NaN,-6.252946,107.010704,NaN,NaN,NaN
8,osm-node-13185499347,NaN,-6.266072,107.008519,NaN,NaN,NaN
9,osm-node-13293910280,Polytron,-6.618346,106.814609,Polytron,NaN,2


In [9]:
# Operator dan kelengkapan tag (dataset kecil, 13 node)
osm["operator"].value_counts(dropna=False)

operator
NaN                          9
BPPT                         1
SPKLU PLN                    1
Polytron                     1
Perusahaan Listrik Negara    1
Name: count, dtype: int64